# SalesPulse - 2.36 Anomaly Detection & Risk Identification
## Continuous KPI Anomaly Monitoring & Alerting System

This notebook demonstrates the end-to-end Anomaly Detection & Risk Identification Workflow for SalesPulse, combining static threshold-based alert rules and dynamic statistical Z-score monitoring.

### Workflow Steps:
1. **Task 1: Threshold-Based Anomaly Detection** - Evaluate business min/max threshold rules across 3+ core metrics.
2. **Task 2: Statistical Anomaly Detection with Z-Score** - Calculate rolling mean and standard deviation over a 30-day lookback window.
3. **Task 3: Severity Classification** - Classify anomalies into `CRITICAL`, `HIGH`, `MEDIUM`, and `LOW` severity levels.
4. **Task 4: Anomaly Logging and Audit Trail** - Store all detected anomalies into a persistent audit log (`anomalies_log.csv`).
5. **Task 5: Visualization with Flagged Points** - Plot raw time-series, 7-day MA, shaded ±2σ expected band, and red 'X' anomaly markers.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot styles
sns.set_theme(style='whitegrid')
plt.rcParams['font.size'] = 11

# Import custom pipeline functions
sys.path.append('..')
from scripts.anomaly_detection_risk import (
    generate_kpi_timeseries,
    task_1_threshold_detection,
    task_2_statistical_zscore,
    task_3_severity_classification,
    task_4_anomaly_logging,
    task_5_visualization
)

### Step 0: Load / Ingest KPI Daily Time Series

In [ ]:
df = generate_kpi_timeseries(num_days=60, seed=42)
print(f"Loaded {len(df)} daily KPI records.")
df.head(10)

### Task 1: Threshold-Based Anomaly Detection

In [ ]:
alert_rules, check_thresholds_fn = task_1_threshold_detection()

### Task 2: Statistical Anomaly Detection with Z-Score

In [ ]:
daily_revenue, anomalies, z_scores, mean, std = task_2_statistical_zscore(df, lookback_days=30, z_threshold=2.0)

### Task 3: Severity Classification

In [ ]:
severity_df, critical_high, classify_severity_fn = task_3_severity_classification(daily_revenue, anomalies, z_scores, mean, std)

### Task 4: Anomaly Logging and Audit Trail

In [ ]:
anomalies_df = task_4_anomaly_logging(daily_revenue, anomalies, z_scores, mean, std, classify_severity_fn)

### Task 5: Diagnostic Visualizations

In [ ]:
task_5_visualization(daily_revenue, anomalies, mean, std)
from IPython.display import Image, display
display(Image('../anomaly_detection.png'))
display(Image('../output/anomaly_severity_distribution.png'))